# Post 2 — Insurance Pricing: An Actuary's Stress Test

**Tabular Foundation Models in the Real World** | Post 2 of 4

This notebook runs the full insurance experiment pipeline:
1. Data loading & validation
2. Feature engineering (raw vs engineered pipelines)
3. Baseline models (GLM, XGBoost, LightGBM) — TABFM-006
4. TabPFN experiments — TABFM-007
5. Statistical comparison
6. Auction / business value analysis
7. Deployment benchmarks

**Before running:** ensure HuggingFace auth is set up for TabPFN:
```bash
hf auth login
# then accept terms at huggingface.co/Prior-Labs/tabpfn_2_6
```

Steps 1–4 (data + baselines) run without GPU. TabPFN at 50K may be slow on CPU — see cell notes.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import yaml
import warnings
warnings.filterwarnings('ignore')

from src.data.load_insurance import load_processed, get_dev_holdout, describe_dataset
from src.features.insurance_features import RawFeaturePipeline, EngineeredFeaturePipeline, GBMFeaturePipeline
from src.models.glm import PoissonGLM, GammaGLM, TweedieGLM
from src.models.gbm import XGBoostModel, LightGBMModel
from src.models.tabpfn_wrapper import TabPFNWrapper, TweedieRecalibrator, IsotonicRecalibrator
from src.evaluation.cv_engine import run_cv, holdout_evaluation
from src.evaluation.auction import run_all_auctions, auction_analysis
from src.evaluation.statistical_tests import compare_models_cv, metric_disagreement_table
from src.metrics import insurance_metrics
from src.plotting import plot_model_comparison, plot_metrics_heatmap

with open('../configs/experiment_config.yaml') as f:
    CONFIG = yaml.safe_load(f)

print('Imports OK')
print(f"Config: {CONFIG['modeling']['approaches']} approaches, {CONFIG['data']['n_cv_folds']} CV folds")

Imports OK
Config: ['freqsev', 'tweedie', 'hurdle'] approaches, 5 CV folds


## Step 1 — Data Loading & Validation

In [2]:
# Load processed dataset (downloads + joins freq & severity if not cached)
df = load_processed()
describe_dataset(df)

Shape:           (678013, 17)
Claim rate:      5.02% (34,060 policies with claims)
Mean exposure:   0.529

ClaimNb distribution:
claim_freq
0    643953
1     32178
2      1784
3        82
4        16
Name: count, dtype: int64

Pure premium (non-zero): mean=10417.8, p50=1504.2, p95=18909.8
Severity (claims>0):     mean=2222.2, p50=1172.0, p95=4659.5

Missing: {'severity': 653069}


In [3]:
# Create splits (saved to disk — identical for all experiments)
splits = get_dev_holdout(df)

X_dev, X_holdout      = splits['X_dev'], splits['X_holdout']
y_dev, y_holdout      = splits['y_dev'], splits['y_holdout']
exp_dev, exp_holdout  = splits['exposure_dev'], splits['exposure_holdout']
cv_folds              = splits['cv_folds']

print(f"Dev: {len(X_dev):,} | Holdout: {len(X_holdout):,}")
print(f"Claim rate — dev: {y_dev.claim_flag.mean():.2%} | holdout: {y_holdout.claim_flag.mean():.2%}")
print(f"CV folds: {len(cv_folds)} × ~{len(X_dev)//len(cv_folds):,} val rows")

Dev: 542,410 | Holdout: 135,603
Claim rate — dev: 5.02% | holdout: 5.02%
CV folds: 5 × ~108,482 val rows


## Step 2 — Feature Pipeline Smoke Test

In [4]:
# Test all three pipelines on a small sample
sample_idx = np.arange(1000)
X_sample = X_dev.iloc[sample_idx]
y_sample = y_dev['pure_premium'].iloc[sample_idx]

raw_pipe = RawFeaturePipeline()
X_raw = raw_pipe.fit_transform(X_sample)
print(f"Raw pipeline:        {X_raw.shape}, dtype={X_raw.dtype}")
print(f"  Features: {raw_pipe.feature_names_}")

eng_pipe = EngineeredFeaturePipeline()
X_eng = eng_pipe.fit_transform(X_sample, y=y_sample)
print(f"\nEngineered pipeline: {X_eng.shape}, dtype={X_eng.dtype}")
print(f"  Features: {eng_pipe.feature_names_}")

gbm_pipe = GBMFeaturePipeline()
X_gbm = gbm_pipe.fit_transform(X_sample, y=y_sample)
print(f"\nGBM pipeline:        {X_gbm.shape}, dtype={X_gbm.dtype}")
print(f"  Features: {gbm_pipe.feature_names_}")

# Verify transform works on unseen data (simulates val fold)
X_val_raw = raw_pipe.transform(X_dev.iloc[1000:1100])
X_val_eng = eng_pipe.transform(X_dev.iloc[1000:1100])
X_val_gbm = gbm_pipe.transform(X_dev.iloc[1000:1100])
print(f"\nTransform on unseen data: raw={X_val_raw.shape}, eng={X_val_eng.shape}, gbm={X_val_gbm.shape} — OK")

# Spot-check GBM business features on a few rows
check = pd.DataFrame(X_gbm, columns=gbm_pipe.feature_names_)
print("\nGBM feature sample (first 3 rows):")
print(check[['BonusMalus', 'DrivAge', 'bm_excess', 'is_malus', 'young_driver', 'senior_driver',
             'young_x_power', 'vehicle_value_proxy']].head(3).to_string())

Raw pipeline:        (1000, 9), dtype=float32
  Features: ['Area', 'VehBrand', 'VehGas', 'Region', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'Density']

Engineered pipeline: (1000, 11), dtype=float32
  Features: ['bm_bin', 'pow_group', 'age_band', 'log_density', 'vehage_bin', 'is_diesel', 'area_ordinal', 'Region_te', 'VehBrand_te', 'age_x_power', 'bm_x_age']

GBM pipeline:        (1000, 15), dtype=float32
  Features: ['VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'log_density', 'area_ordinal', 'is_diesel', 'Region_te', 'VehBrand_te', 'bm_excess', 'is_malus', 'young_driver', 'senior_driver', 'young_x_power', 'vehicle_value_proxy']

Transform on unseen data: raw=(100, 9), eng=(100, 11), gbm=(100, 15) — OK

GBM feature sample (first 3 rows):
   BonusMalus  DrivAge  bm_excess  is_malus  young_driver  senior_driver  young_x_power  vehicle_value_proxy
0        50.0     55.0        0.0       0.0           0.0            0.0            0.0                  5.0
1        50.0     55.0     

## Step 3 — GLM Baselines (TABFM-006)

GLMs are the industry standard. They run on the full dataset (no subsampling needed).
We use a small sample here for smoke-testing; the full CV run is in Step 5.

In [5]:
# Quick GLM smoke test on 5K sample
N_SMOKE = 5_000
idx = np.arange(N_SMOKE)
X_tr_sm = X_dev.iloc[idx]
y_tr_sm = y_dev.iloc[idx]
exp_tr_sm = exp_dev.iloc[idx].values

raw_pipe_sm = RawFeaturePipeline()
X_raw_sm = raw_pipe_sm.fit_transform(X_tr_sm)

# Poisson frequency
glm_poi = PoissonGLM()
glm_poi.fit(X_raw_sm, y_tr_sm['claim_freq'].values, exposure=exp_tr_sm)
pred_rate = glm_poi.predict(X_raw_sm, exposure=exp_tr_sm)
print(f"GLM Poisson — mean predicted rate: {pred_rate.mean():.4f} (actual: {(y_tr_sm.claim_freq / exp_tr_sm).mean():.4f})")

# Tweedie pure premium
glm_tw = TweedieGLM()
glm_tw.fit(X_raw_sm, y_tr_sm['pure_premium'].values, exposure=exp_tr_sm)
pred_pp = glm_tw.predict(X_raw_sm, exposure=exp_tr_sm)
print(f"GLM Tweedie — mean predicted PP: {pred_pp.mean():.2f} (actual: {y_tr_sm.pure_premium.mean():.2f})")

# Gamma severity (claims with positive amount only)
# severity is NaN where ClaimNb=0 OR ClaimAmount_total=0 — use notna() mask for both X and y
sev_mask = y_tr_sm['severity'].notna().values
if sev_mask.sum() > 10:
    glm_gamma = GammaGLM()
    glm_gamma.fit(X_raw_sm[sev_mask], y_tr_sm['severity'].dropna().values)
    print(f"GLM Gamma — fit on {sev_mask.sum()} claims — OK")

print("\nGLM smoke tests: PASS")

GLM Poisson — mean predicted rate: 3.0490 (actual: 13.4170)
GLM Tweedie — mean predicted PP: 1146.09 (actual: 178.25)
GLM Gamma — fit on 119 claims — OK

GLM smoke tests: PASS


## Step 4 — GBM Hyperparameter Tuning (TABFM-006)

Tuning runs ONCE on the dev set (not per outer fold).
Best params are cached to `configs/`. Skip tuning if cached params exist.

In [6]:
# Run on a subsample for local development; use full dev for final runs
TUNE_N = 50_000  # increase to len(X_dev) for production tuning
tune_idx = np.random.default_rng(42).choice(len(X_dev), size=TUNE_N, replace=False)

X_tune_raw = RawFeaturePipeline().fit_transform(X_dev.iloc[tune_idx])
X_tune_eng = EngineeredFeaturePipeline().fit_transform(
    X_dev.iloc[tune_idx],
    y=y_dev['pure_premium'].iloc[tune_idx]
)
X_tune_gbm = GBMFeaturePipeline().fit_transform(
    X_dev.iloc[tune_idx],
    y=y_dev['pure_premium'].iloc[tune_idx]
)
y_tune_tw  = y_dev['pure_premium'].iloc[tune_idx].values
exp_tune   = exp_dev.iloc[tune_idx].values

print(f"Tuning on {TUNE_N:,} rows")
print(f"  raw={X_tune_raw.shape}, eng={X_tune_eng.shape}, gbm={X_tune_gbm.shape}")

# XGBoost Tweedie — raw features
xgb_tw_raw = XGBoostModel(approach='tweedie')
xgb_tw_raw.tune(X_tune_raw, y_tune_tw, exposure_dev=exp_tune,
                n_trials=CONFIG['tuning']['n_trials'],
                features_label='raw')

# XGBoost Tweedie — engineered features
xgb_tw_eng = XGBoostModel(approach='tweedie')
xgb_tw_eng.tune(X_tune_eng, y_tune_tw, exposure_dev=exp_tune,
                n_trials=CONFIG['tuning']['n_trials'],
                features_label='engineered')

# XGBoost Tweedie — GBM features
xgb_tw_gbm = XGBoostModel(approach='tweedie')
xgb_tw_gbm.tune(X_tune_gbm, y_tune_tw, exposure_dev=exp_tune,
                n_trials=CONFIG['tuning']['n_trials'],
                features_label='gbm')

# LightGBM Tweedie — raw
lgbm_tw_raw = LightGBMModel(approach='tweedie')
lgbm_tw_raw.tune(X_tune_raw, y_tune_tw, exposure_dev=exp_tune,
                 n_trials=CONFIG['tuning']['n_trials'],
                 features_label='raw')

# LightGBM Tweedie — engineered
lgbm_tw_eng = LightGBMModel(approach='tweedie')
lgbm_tw_eng.tune(X_tune_eng, y_tune_tw, exposure_dev=exp_tune,
                 n_trials=CONFIG['tuning']['n_trials'],
                 features_label='engineered')

# LightGBM Tweedie — GBM features
lgbm_tw_gbm = LightGBMModel(approach='tweedie')
lgbm_tw_gbm.tune(X_tune_gbm, y_tune_tw, exposure_dev=exp_tune,
                 n_trials=CONFIG['tuning']['n_trials'],
                 features_label='gbm')

print("\nTuning complete. Params cached to configs/")

Tuning on 50,000 rows
  raw=(50000, 9), eng=(50000, 11), gbm=(50000, 15)
Loaded cached XGB params from /Users/alexruppelt/Documents/Projects/tabpfn_analysis/notebooks/../configs/xgb_best_params_tweedie_raw.json
Loaded cached XGB params from /Users/alexruppelt/Documents/Projects/tabpfn_analysis/notebooks/../configs/xgb_best_params_tweedie_engineered.json


  0%|          | 0/50 [00:00<?, ?it/s]

Best XGB params (tweedie, gbm): {'max_depth': 3, 'learning_rate': 0.0127627037205584, 'n_estimators': 101, 'min_child_weight': 73, 'subsample': 0.8506048699779767, 'colsample_bytree': 0.6247606607030806, 'reg_alpha': 1.2201423458201578e-06, 'reg_lambda': 1.0705558215905238}
Loaded cached LGBM params from /Users/alexruppelt/Documents/Projects/tabpfn_analysis/notebooks/../configs/lgbm_best_params_tweedie_raw.json
Loaded cached LGBM params from /Users/alexruppelt/Documents/Projects/tabpfn_analysis/notebooks/../configs/lgbm_best_params_tweedie_engineered.json


  0%|          | 0/50 [00:00<?, ?it/s]

Best LGBM params (tweedie, gbm): {'num_leaves': 22, 'learning_rate': 0.013800488261266623, 'n_estimators': 101, 'min_child_samples': 57, 'subsample': 0.7585126522395964, 'colsample_bytree': 0.8956749320543134, 'reg_alpha': 2.993017981471946e-05, 'reg_lambda': 0.004385057593084603}

Tuning complete. Params cached to configs/


## Step 5 — Full 5-Fold CV: Baselines (TABFM-006)

Approach B (direct Tweedie) for all models as the primary comparison.
Approaches A and C added after.

In [7]:
# GLM Tweedie — raw features
glm_results_raw = run_cv(
    model_factory=TweedieGLM,
    feature_pipeline_factory=RawFeaturePipeline,
    X_dev=X_dev,
    y_dev=y_dev,
    exposure_dev=exp_dev,
    cv_folds=cv_folds,
    approach='tweedie',
    model_name='GLM_Tweedie',
    features_label='raw',
)
print("GLM Tweedie raw mean metrics:", glm_results_raw['mean_metrics'])

GLM_Tweedie/raw/tweedie:  20%|██        | 1/5 [00:02<00:09,  2.49s/it]

  Fold 0: tweedie=184.7991 gini=0.1958


GLM_Tweedie/raw/tweedie:  40%|████      | 2/5 [00:04<00:06,  2.30s/it]

  Fold 1: tweedie=187.8718 gini=0.3089


GLM_Tweedie/raw/tweedie:  60%|██████    | 3/5 [00:06<00:04,  2.30s/it]

  Fold 2: tweedie=182.3157 gini=0.4723


GLM_Tweedie/raw/tweedie:  80%|████████  | 4/5 [00:09<00:02,  2.58s/it]

  Fold 3: tweedie=187.5409 gini=0.3709


GLM_Tweedie/raw/tweedie: 100%|██████████| 5/5 [00:12<00:00,  2.43s/it]

  Fold 4: tweedie=180.9558 gini=0.4956
GLM Tweedie raw mean metrics: {'tweedie_dev_1.5': 184.6966565965215, 'poisson_dev': 6079.209851441598, 'gini': 0.36869637354453416, 'rmse': 15885.075804353683, 'mae': 2841.6110986580707}


In [8]:
# GLM Tweedie — engineered features
glm_results_eng = run_cv(
    model_factory=TweedieGLM,
    feature_pipeline_factory=EngineeredFeaturePipeline,
    X_dev=X_dev,
    y_dev=y_dev,
    exposure_dev=exp_dev,
    cv_folds=cv_folds,
    approach='tweedie',
    model_name='GLM_Tweedie',
    features_label='engineered',
)
print("GLM Tweedie engineered mean metrics:", glm_results_eng['mean_metrics'])

GLM_Tweedie/engineered/tweedie:  20%|██        | 1/5 [00:02<00:10,  2.51s/it]

  Fold 0: tweedie=182.3605 gini=0.1382


GLM_Tweedie/engineered/tweedie:  40%|████      | 2/5 [00:04<00:07,  2.41s/it]

  Fold 1: tweedie=186.6199 gini=0.2778


GLM_Tweedie/engineered/tweedie:  60%|██████    | 3/5 [00:07<00:04,  2.44s/it]

  Fold 2: tweedie=179.2498 gini=0.4105


GLM_Tweedie/engineered/tweedie:  80%|████████  | 4/5 [00:09<00:02,  2.49s/it]

  Fold 3: tweedie=188.8242 gini=0.4172


GLM_Tweedie/engineered/tweedie: 100%|██████████| 5/5 [00:11<00:00,  2.33s/it]

  Fold 4: tweedie=179.7852 gini=0.4727
GLM Tweedie engineered mean metrics: {'tweedie_dev_1.5': 183.36790421121322, 'poisson_dev': 5982.7064190882065, 'gini': 0.3433001078394038, 'rmse': 15655.616121816289, 'mae': 2780.999949547104}


In [9]:
# GLM Tweedie — raw features
glm_results_raw = run_cv(
    model_factory=TweedieGLM,
    feature_pipeline_factory=RawFeaturePipeline,
    X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
    cv_folds=cv_folds, approach='tweedie',
    model_name='GLM_Tweedie', features_label='raw',
)

# GLM Tweedie — engineered features
glm_results_eng = run_cv(
    model_factory=TweedieGLM,
    feature_pipeline_factory=EngineeredFeaturePipeline,
    X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
    cv_folds=cv_folds, approach='tweedie',
    model_name='GLM_Tweedie', features_label='engineered',
)

# XGBoost Tweedie — raw
xgb_cv_raw = run_cv(
    model_factory=lambda: XGBoostModel(approach='tweedie'),
    feature_pipeline_factory=RawFeaturePipeline,
    X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
    cv_folds=cv_folds, approach='tweedie',
    model_name='XGBoost', features_label='raw',
)

# XGBoost Tweedie — engineered
xgb_cv_eng = run_cv(
    model_factory=lambda: XGBoostModel(approach='tweedie'),
    feature_pipeline_factory=EngineeredFeaturePipeline,
    X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
    cv_folds=cv_folds, approach='tweedie',
    model_name='XGBoost', features_label='engineered',
)

# XGBoost Tweedie — GBM-optimised features
xgb_cv_gbm = run_cv(
    model_factory=lambda: XGBoostModel(approach='tweedie'),
    feature_pipeline_factory=GBMFeaturePipeline,
    X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
    cv_folds=cv_folds, approach='tweedie',
    model_name='XGBoost', features_label='gbm',
)

# LightGBM Tweedie — raw
lgbm_cv_raw = run_cv(
    model_factory=lambda: LightGBMModel(approach='tweedie'),
    feature_pipeline_factory=RawFeaturePipeline,
    X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
    cv_folds=cv_folds, approach='tweedie',
    model_name='LightGBM', features_label='raw',
)

# LightGBM Tweedie — engineered
lgbm_cv_eng = run_cv(
    model_factory=lambda: LightGBMModel(approach='tweedie'),
    feature_pipeline_factory=EngineeredFeaturePipeline,
    X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
    cv_folds=cv_folds, approach='tweedie',
    model_name='LightGBM', features_label='engineered',
)

# LightGBM Tweedie — GBM-optimised features
lgbm_cv_gbm = run_cv(
    model_factory=lambda: LightGBMModel(approach='tweedie'),
    feature_pipeline_factory=GBMFeaturePipeline,
    X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
    cv_folds=cv_folds, approach='tweedie',
    model_name='LightGBM', features_label='gbm',
)

print("\n=== CV Summary (Tweedie deviance, lower is better) ===")
cv_summary = pd.DataFrame({
    'GLM_raw':      glm_results_raw['mean_metrics'],
    'GLM_eng':      glm_results_eng['mean_metrics'],
    'XGB_raw':      xgb_cv_raw['mean_metrics'],
    'XGB_eng':      xgb_cv_eng['mean_metrics'],
    'XGB_gbm':      xgb_cv_gbm['mean_metrics'],
    'LGBM_raw':     lgbm_cv_raw['mean_metrics'],
    'LGBM_eng':     lgbm_cv_eng['mean_metrics'],
    'LGBM_gbm':     lgbm_cv_gbm['mean_metrics'],
}).T
print(cv_summary.round(4))

print("\n--- Hypothesis check: GBM pipeline should beat both raw and engineered for XGB/LGBM ---")
for model_name, raw, eng, gbm in [
    ("XGBoost",  xgb_cv_raw,  xgb_cv_eng,  xgb_cv_gbm),
    ("LightGBM", lgbm_cv_raw, lgbm_cv_eng, lgbm_cv_gbm),
]:
    td_raw = raw['mean_metrics'].get('tweedie_dev_1.5', float('nan'))
    td_eng = eng['mean_metrics'].get('tweedie_dev_1.5', float('nan'))
    td_gbm = gbm['mean_metrics'].get('tweedie_dev_1.5', float('nan'))
    best = min(td_raw, td_eng, td_gbm)
    winner = ['raw', 'eng', 'gbm'][[td_raw, td_eng, td_gbm].index(best)]
    print(f"  {model_name}: raw={td_raw:.4f}, eng={td_eng:.4f}, gbm={td_gbm:.4f} → best={winner}")

GLM_Tweedie/raw/tweedie:  20%|██        | 1/5 [00:02<00:08,  2.21s/it]

  Fold 0: tweedie=184.7991 gini=0.1958


GLM_Tweedie/raw/tweedie:  40%|████      | 2/5 [00:04<00:06,  2.16s/it]

  Fold 1: tweedie=187.8718 gini=0.3089


GLM_Tweedie/raw/tweedie:  60%|██████    | 3/5 [00:06<00:04,  2.14s/it]

...

  Fold 2: tweedie=75.6246 gini=0.5207


LightGBM/gbm/tweedie:  80%|████████  | 4/5 [00:04<00:01,  1.12s/it]

  Fold 3: tweedie=79.2225 gini=0.4382


LightGBM/gbm/tweedie: 100%|██████████| 5/5 [00:05<00:00,  1.13s/it]

  Fold 4: tweedie=78.4690 gini=0.5539

=== CV Summary (Tweedie deviance, lower is better) ===
          tweedie_dev_1.5  poisson_dev    gini        rmse        mae
GLM_raw          184.6967    6079.2099  0.3687  15885.0758  2841.6111
GLM_eng          183.3679    5982.7064  0.3433  15655.6161  2780.9999
XGB_raw           89.7785    1781.1501  0.3433  14503.9237   260.1104
XGB_eng           92.5775    1840.8781  0.3688  14602.0085   273.7805
XGB_gbm           91.1937    1762.6882  0.4172  14504.1355   259.5507
LGBM_raw          80.0857    1685.3422  0.4001  14498.1855   272.9102
LGBM_eng          83.6347    1766.6254  0.4193  14512.2474   280.8375
LGBM_gbm          79.5486    1664.4111  0.4343  14497.7416   272.5182

--- Hypothesis check: GBM pipeline should beat both raw and engineered for XGB/LGBM ---
  XGBoost: raw=89.7785, eng=92.5775, gbm=91.1937 → best=raw
  LightGBM: raw=80.0857, eng=83.6347, gbm=79.5486 → best=gbm


## Step 6 — TabPFN Experiments (TABFM-007)

**Requirements:**
- HuggingFace auth must be set up (`hf auth login`)
- 10K subsample: runs locally on CPU (~2-5 min per fold)
- 50K subsample: slow on CPU — consider running on GPU/cloud

TabPFN API constraints documented:
- No `sample_weight` → exposure via `rate` or `feature` strategy
- No Tweedie objective → recalibration applied (Option B: nested inner-fold OOF)

In [14]:
import huggingface_hub
huggingface_hub.login()

In [15]:
# Verify TabPFN auth before running the full experiment
try:
    from tabpfn import TabPFNRegressor
    # Try initialising — this triggers model download
    r = TabPFNRegressor()
    print("TabPFN auth: OK")
    TABPFN_AVAILABLE = True
except Exception as e:
    print(f"TabPFN auth FAILED: {e}")
    print("Run: hf auth login")
    TABPFN_AVAILABLE = False

TabPFN auth: OK


In [ ]:
if TABPFN_AVAILABLE:
    # TabPFN 10K — rate strategy — raw features
    tabpfn_10k_raw = run_cv(
        model_factory=lambda: TabPFNWrapper(task='regression', n_train_max=10_000, exposure_strategy='rate'),
        feature_pipeline_factory=RawFeaturePipeline,
        X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
        cv_folds=cv_folds, approach='tweedie',
        model_name='TabPFN_10K_rate', features_label='raw',
        tabpfn_n_inner_folds=3,
    )

    # TabPFN 10K — rate strategy — engineered
    tabpfn_10k_eng = run_cv(
        model_factory=lambda: TabPFNWrapper(task='regression', n_train_max=10_000, exposure_strategy='rate'),
        feature_pipeline_factory=EngineeredFeaturePipeline,
        X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
        cv_folds=cv_folds, approach='tweedie',
        model_name='TabPFN_10K_rate', features_label='engineered',
        tabpfn_n_inner_folds=3,
    )

    print("TabPFN 10K CV done")
else:
    print("Skipping TabPFN — auth not set up")

TabPFN_10K_rate/raw/tweedie:   0%|          | 0/5 [00:00<?, ?it/s]

tabpfn-v2.6-regressor-v2.6_default.ckpt:   0%|          | 0.00/51.6M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

In [ ]:
# TabPFN 50K — slower, consider running on GPU
# Uncomment when compute is available

# if TABPFN_AVAILABLE:
#     tabpfn_50k_raw = run_cv(
#         model_factory=lambda: TabPFNWrapper(task='regression', n_train_max=50_000, exposure_strategy='rate'),
#         feature_pipeline_factory=RawFeaturePipeline,
#         X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
#         cv_folds=cv_folds, approach='tweedie',
#         model_name='TabPFN_50K_rate', features_label='raw',
#     )
#     tabpfn_50k_eng = run_cv(
#         model_factory=lambda: TabPFNWrapper(task='regression', n_train_max=50_000, exposure_strategy='rate'),
#         feature_pipeline_factory=EngineeredFeaturePipeline,
#         X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
#         cv_folds=cv_folds, approach='tweedie',
#         model_name='TabPFN_50K_rate', features_label='engineered',
#     )

## Step 7 — Hurdle Model (Approach C)

Stage 1 = TabPFNClassifier (its natural turf)
Stage 2 = GammaGLM / TabPFNRegressor on claims-only (~15-20K rows — within TabPFN's limit)

In [ ]:
if TABPFN_AVAILABLE:
    # Hurdle stage 1: classification
    tabpfn_hurdle_cls = run_cv(
        model_factory=lambda: TabPFNWrapper(task='classification', n_train_max=10_000),
        feature_pipeline_factory=RawFeaturePipeline,
        X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
        cv_folds=cv_folds, approach='hurdle',
        model_name='TabPFN_10K_hurdle_cls', features_label='raw',
    )
    print("Hurdle classification CV done")

## Step 8 — Statistical Comparison of CV Results

In [ ]:
from src.evaluation.statistical_tests import compare_models_cv, rank_models_by_metric

# Primary comparison: does the GBM pipeline beat raw for each model?
comparisons = [
    # Feature pipeline effect within models
    (xgb_cv_gbm['fold_metrics'],  xgb_cv_raw['fold_metrics'],  'XGB_gbm',  'XGB_raw'),
    (xgb_cv_gbm['fold_metrics'],  xgb_cv_eng['fold_metrics'],  'XGB_gbm',  'XGB_eng'),
    (lgbm_cv_gbm['fold_metrics'], lgbm_cv_raw['fold_metrics'], 'LGBM_gbm', 'LGBM_raw'),
    (lgbm_cv_gbm['fold_metrics'], lgbm_cv_eng['fold_metrics'], 'LGBM_gbm', 'LGBM_eng'),
    # Best GBM vs best GLM
    (xgb_cv_gbm['fold_metrics'],  glm_results_eng['fold_metrics'], 'XGB_gbm',  'GLM_eng'),
    (lgbm_cv_gbm['fold_metrics'], glm_results_eng['fold_metrics'], 'LGBM_gbm', 'GLM_eng'),
]

if TABPFN_AVAILABLE:
    comparisons.append(
        (tabpfn_10k_eng['fold_metrics'], xgb_cv_gbm['fold_metrics'], 'TabPFN_10K_eng', 'XGB_gbm')
    )

for fold_a, fold_b, name_a, name_b in comparisons:
    result = compare_models_cv(fold_a, fold_b, metric='tweedie_dev_1.5', name_a=name_a, name_b=name_b)
    print(result['interpretation'])
    print()

# Full ranking table
all_fold_metrics = {
    'GLM_raw':   glm_results_raw['fold_metrics'],
    'GLM_eng':   glm_results_eng['fold_metrics'],
    'XGB_raw':   xgb_cv_raw['fold_metrics'],
    'XGB_eng':   xgb_cv_eng['fold_metrics'],
    'XGB_gbm':   xgb_cv_gbm['fold_metrics'],
    'LGBM_raw':  lgbm_cv_raw['fold_metrics'],
    'LGBM_eng':  lgbm_cv_eng['fold_metrics'],
    'LGBM_gbm':  lgbm_cv_gbm['fold_metrics'],
}
print("\n=== CV Rankings (Tweedie deviance, lower is better) ===")
print(rank_models_by_metric(all_fold_metrics, metric='tweedie_dev_1.5').round(4))

## Step 9 — Holdout Evaluation

In [ ]:
model_configs = [
    {'model_factory': TweedieGLM,
     'pipeline_factory': RawFeaturePipeline,
     'approach': 'tweedie', 'model_name': 'GLM_Tweedie', 'features_label': 'raw'},
    {'model_factory': TweedieGLM,
     'pipeline_factory': EngineeredFeaturePipeline,
     'approach': 'tweedie', 'model_name': 'GLM_Tweedie', 'features_label': 'engineered'},
    {'model_factory': lambda: XGBoostModel(approach='tweedie'),
     'pipeline_factory': RawFeaturePipeline,
     'approach': 'tweedie', 'model_name': 'XGBoost', 'features_label': 'raw'},
    {'model_factory': lambda: XGBoostModel(approach='tweedie'),
     'pipeline_factory': EngineeredFeaturePipeline,
     'approach': 'tweedie', 'model_name': 'XGBoost', 'features_label': 'engineered'},
    {'model_factory': lambda: XGBoostModel(approach='tweedie'),
     'pipeline_factory': GBMFeaturePipeline,
     'approach': 'tweedie', 'model_name': 'XGBoost', 'features_label': 'gbm'},
    {'model_factory': lambda: LightGBMModel(approach='tweedie'),
     'pipeline_factory': RawFeaturePipeline,
     'approach': 'tweedie', 'model_name': 'LightGBM', 'features_label': 'raw'},
    {'model_factory': lambda: LightGBMModel(approach='tweedie'),
     'pipeline_factory': EngineeredFeaturePipeline,
     'approach': 'tweedie', 'model_name': 'LightGBM', 'features_label': 'engineered'},
    {'model_factory': lambda: LightGBMModel(approach='tweedie'),
     'pipeline_factory': GBMFeaturePipeline,
     'approach': 'tweedie', 'model_name': 'LightGBM', 'features_label': 'gbm'},
]

holdout_results = holdout_evaluation(
    model_configs=model_configs,
    X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
    X_holdout=X_holdout, y_holdout=y_holdout, exposure_holdout=exp_holdout,
)
print(holdout_results.round(4))

## Step 10 — Metric Disagreement Table

In [ ]:
from src.evaluation.statistical_tests import metric_disagreement_table

ranks = metric_disagreement_table(holdout_results)
print("\nModel ranks by metric (1 = best):")
print(ranks)

## Step 11 — Auction Analysis

In [ ]:
from src.evaluation.auction import run_all_auctions

auction_results = run_all_auctions(
    holdout_results=holdout_results,
    y_holdout=y_holdout['pure_premium'].values,
    exposure_holdout=exp_holdout.values,
)

for matchup, res in auction_results.items():
    agg = res['aggregate']
    print(f"\n{matchup}")
    print(f"  {agg['model_a']}: wins {agg['a_pct_policies']:.1%}, loss ratio {agg['a_loss_ratio']:.3f}, profit {agg['a_profit']:,.0f}")
    print(f"  {agg['model_b']}: wins {agg['b_pct_policies']:.1%}, loss ratio {agg['b_loss_ratio']:.3f}, profit {agg['b_profit']:,.0f}")

## Step 12 — Key Visualisations

In [ ]:
import matplotlib.pyplot as plt
from src.plotting import plot_model_comparison, plot_metrics_heatmap

# Tweedie deviance comparison
fig, ax = plot_model_comparison(
    holdout_results,
    metric='tweedie_dev_1.5',
    title='Tweedie deviance by model (holdout, lower is better)',
)
fig.savefig('../figures/post2/tweedie_deviance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Multi-metric heatmap
fig2, ax2 = plot_metrics_heatmap(
    holdout_results,
    metrics=['tweedie_dev_1.5', 'poisson_dev', 'gini', 'rmse', 'mae'],
    title='Model × metric comparison (holdout)',
)
fig2.savefig('../figures/post2/metric_disagreement_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 13 — Deployment Benchmarks (TABFM-010)

In [ ]:
import time
import tracemalloc
import pickle

# Latency benchmark at 10K rows
benchmark_models = {
    'GLM_Tweedie': (glm_results_eng, TweedieGLM()),
    'XGBoost':     (xgb_cv_eng, XGBoostModel(approach='tweedie')),
    'LightGBM':    (lgbm_cv_eng, LightGBMModel(approach='tweedie')),
}

bench_rows = []
X_bench = EngineeredFeaturePipeline().fit_transform(
    X_holdout.head(10_000),
    y=y_holdout['pure_premium'].head(10_000)
)

for name, (cv_res, model) in benchmark_models.items():
    # Fit on small sample for benchmark purposes
    pipe = EngineeredFeaturePipeline()
    X_fit = pipe.fit_transform(X_dev.head(50_000), y=y_dev['pure_premium'].head(50_000))
    model.fit(X_fit, y_dev['pure_premium'].head(50_000).values, exposure=exp_dev.head(50_000).values)
    
    # Time inference
    times = []
    for _ in range(5):
        t0 = time.perf_counter()
        model.predict(X_bench)
        times.append(time.perf_counter() - t0)
    
    # Memory
    tracemalloc.start()
    model.predict(X_bench)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    # Model size
    size_mb = len(pickle.dumps(model)) / 1e6
    
    bench_rows.append({
        'model': name,
        'median_latency_s': round(np.median(times), 4),
        'peak_memory_mb': round(peak / 1e6, 1),
        'model_size_mb': round(size_mb, 2),
    })

bench_df = pd.DataFrame(bench_rows).set_index('model')
print(bench_df)
bench_df.to_parquet('../results/deployment/benchmarks.parquet')